# 第 1 周末练习 —— 技术问答解释器（GPT + Ollama）

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、准确的解释（短段落或要点均可）
- **对比**：同一问题分别问云端 `gpt-4o-mini`（流式）与本地 `llama3.2`（一次性返回）

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai_client.chat.completions.create(...)` |
| `messages`（system / user） | `SYSTEM_PROMPT` + `question` |
| 流式输出 `stream=True` | GPT 路径逐块 `print(..., end="", flush=True)` |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），经 OpenAI 兼容 `/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：必须有 `OPENAI_API_KEY`；缺了会在环境设置格直接 `raise ValueError`
3. 确保本机 Ollama 在 `http://localhost:11434` 运行，并已 `ollama pull llama3.2`
4. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格做对比


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os

# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 既可打云端，也可打本地 Ollama 兼容端点
from openai import OpenAI


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境 + 双客户端：云端 OpenAI 与本地 Ollama（OpenAI 兼容） ==========

# load_dotenv(override=True)：读取 .env；override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量读取 OpenAI 密钥；没有就尽早失败，避免后面 API 报一长串难懂错误
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    # 错误文案保留英文：这是面向运行时的提示字符串，按硬规则不翻译
    raise ValueError(
        "Missing OPENAI_API_KEY. Add it to a .env file in the project root "
        "(see week1/day1.ipynb and the troubleshooting notebook if needed)."
    )

# 默认 OpenAI 客户端：内部会使用上面的 OPENAI_API_KEY
openai_client = OpenAI()

# Ollama 的 OpenAI 兼容基址：注意是 /v1，不是原生 /api/chat
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 第二个客户端：base_url 指向本地；api_key 任意非空占位即可（Ollama 通常不校验）
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# system prompt 保留英文：发给模型的角色/风格指令，改译会改变回答行为
SYSTEM_PROMPT = """You are a patient technical tutor. Explain clearly and accurately.
Use short sections or bullet points when helpful. If you show code, keep it minimal."""



In [ ]:
# ========== 提问 + messages：改 question 就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

# messages：system 定「怎么答」，user 放具体问题；strip() 去掉首尾空白
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question.strip()},
]


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# stream=True：持续返回增量 delta，而不是等整段生成完
stream = openai_client.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True,
)

# 打印分隔标题（给人看的 UI 文案；保留原英文/符号风格）
print("— GPT-4o-mini (streaming) —\n")
# 遍历每个流式 chunk，取出增量文本并立刻刷到终端
for chunk in stream:
    # delta.content 可能为 None，用 or "" 避免打印 None
    piece = chunk.choices[0].delta.content or ""
    # end="" 不换行拼接；flush=True 立刻输出，实现「打字机」效果
    print(piece, end="", flush=True)
# 流结束后补一个换行，避免后续输出粘在同一行
print("\n")


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）一次性回答 ==========
# 前提：Ollama 必须运行（原注释写 `ollamaserve`；常见命令是 `ollama serve`）并已 pull llama3.2
# 让 Llama 3.2 回答（Ollama 必须运行：`ollamaserve` 和 `ollama pull llama3.2`）

# 非流式 create：等整段生成完再返回；与上面 GPT 流式路径形成对比
response = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
)

# 打印分隔标题，标明这是本地 Llama 路径
print("— Llama 3.2 via Ollama —\n")
# 取出助手消息正文并整段打印
print(response.choices[0].message.content)
